# The Determinants of Small Shareholder Victory

In [25]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [26]:
%%stata

clear all
set more off
set varabbrev off
version 18


* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [27]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen month = month(day)
gen quarter = quarter(day)

gen year = year(day)
encode gecko_id, gen(token)
encode space,    gen(dao)
encode topic,    gen(proposal_type)

* Dependent variables
local wins non_whale_victory_vn non_whale_victory_vp non_whale_victory_vp_vn

replace concensus = concensus * have_discussion
replace professionalism = professionalism * have_discussion

* Independent variables
local complexity multi_choices weighted quorum ranked_choice
local discussion_char concensus professionalism


* Label variables
label var non_whale_victory_vn "\${\it Small Win}^{vn}_{i,t}\$"
label var non_whale_victory_vp "\${\it Small Win}^{vp}_{i,t}\$"
label var non_whale_victory_vp_vn "\${\it Small Win}^{vp,vn}_{i,t}\$"
label var weighted "\${\it Weighted}_{i,t}\$"
label var quadratic "\${\it Quadratic Voting}_{i,t}\$"
label var ranked_choice "\${\it Ranked Choice}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var professionalism  "\${\it Professionalism}_{i,t}\$"
label var concensus       "\${\it Concensus}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var whale_user          "\${\it User}_{i,t}^{\it Whale}\$"
label var non_whale_participation    "\${\it Participation}_{i,t}^{\it Small}\$"


. 
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(59 vars, 2,830 obs)

. 
. * Parse date
. gen day = date(date, "YMD")

. format day %td

. gen month = month(day)

. gen quarter = quarter(day)

. 
. gen year = year(day)

. encode gecko_id, gen(token)

. encode space,    gen(dao)

. encode topic,    gen(proposal_type)

. 
. * Dependent variables
. local wins non_whale_victory_vn non_whale_victory_vp non_whale_victory_vp_vn

. 
. replace concensus = concensus * have_discussion
(219 real changes made)

. replace professionalism = professionalism * have_discussion
(218 real changes made)

. 
. * Independent variables
. local complexity multi_choices weighted quorum ranked_choice

. local discussion_char concensus professionalism

. 
. 
. * Label variables
. label var non_whale_victory_vn "\${\it Small Win}^{vn}_{i,t}\$"

. label var non_whale_victory_vp "\${\it Small Win}^{vp}_{i,t}\$"

. label var non_whale_victory_vp_

## Proposal Characteristics

In [28]:
%%stata

******************************************************
* LOGISTIC REGRESSIONS
******************************************************

eststo clear

foreach y of local wins {

    * VIF test for multicollinearity
    qui reg `y' `user_char' delegation `complexity' `dapp_char' 
    estat vif

    ppmlhdfe `y' `complexity' delegation have_discussion, absorb(year proposal_type) vce(cluster proposal_type)
    eststo `y'
    estadd local fe_token "Y"
    estadd local fe_time  "Y"

    # * Run regression with SEs clustered by DAO
    # reghdfe `y' `complexity' delegation have_discussion, absorb(token year) vce(cluster dao)
    # eststo `y'
    # estadd local fe_token "Y"
    # estadd local fe_time  "Y"

    # * Run regression with discussion characteristics
    # reghdfe `y' `complexity' delegation concensus professionalism, absorb(token year) vce(cluster dao)
    # eststo `y'_d
    # estadd local fe_token "Y"
    # estadd local fe_time  "Y"

    * Run regression with dapp characteristics
    ppmlhdfe `y' `complexity' delegation have_discussion non_whale_user, absorb(year proposal_type) vce(cluster proposal_type)
    eststo `y'_u
    estadd local fe_token "Y"
    estadd local fe_time  "Y"


}

* Export LaTeX table
esttab                                                     ///
    non_whale_victory_vn  non_whale_victory_vn_u           ///
    non_whale_victory_vp non_whale_victory_vp_u            ///
    non_whale_victory_vp_vn non_whale_victory_vp_vn_u      ///
    using "$TABLES/reg_logit_win_char.tex", replace        ///
    se star(* 0.10 ** 0.05 *** 0.01)                       ///
    b(%9.3f) se(%9.2f) label nogaps nonotes booktabs       ///
    noomitted eqlabels(none) nomtitles nocon               ///
    mgroups("\${\it Small Win}^{vn}_{i,t}\$"              ///
            "\${\it Small Win}^{vp}_{i,t}\$"              ///
            "\${\it Small Win}^{vp,vn}_{i,t}\$",          ///
                span                                      ///
                pattern(1 1 1)               ///
                prefix(\multicolumn{@span}{c}{)           ///
                suffix(})                                 ///
                erepeat(\cmidrule(lr){@span}) )           ///
    substitute("\_" "_")                                   ///
    stats(fe_token fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Token FE" "Year FE" "Observations" "Adjusted R²"))


. 
. ******************************************************
. * LOGISTIC REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
. foreach y of local wins {
  2. 
.     * VIF test for multicollinearity
.     qui reg `y' `user_char' delegation `complexity' `dapp_char' 
  3.     estat vif
  4. 
.     ppmlhdfe `y' `complexity' delegation have_discussion, absorb(year proposa
> l_type) vce(cluster proposal_type)
  5.     eststo `y'
  6.     estadd local fe_token "Y"
  7.     estadd local fe_time  "Y"
  8. 
.     # * Run regression with SEs clustered by DAO
Unknown #command
.     # reghdfe `y' `complexity' delegation have_discussion, absorb(token year)
>  vce(cluster dao)
Unknown #command
.     # eststo `y'
Unknown #command
.     # estadd local fe_token "Y"
Unknown #command
.     # estadd local fe_time  "Y"
Unknown #command
. 
.     # * Run regression with discussion characteristics
Unknown #command
.     # reghdfe `y' `complexity' delegation concensus pro